# Cruzamento das bases (PAM x PPM x PIB)

Integra as três tabelas obrigatórias do Tema 2 (Agropecuária e transformação econômica) pela chave **código IBGE do município** (`territorio_codigo`) + **ano**, no nível `nivel_territorial_nome == 'Município'`. Não usa nome do município como chave, conforme exigido.

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

**Regras de agregação respeitadas neste cruzamento:**
- PAM: quantidades de produtos diferentes (unidades distintas) não são somadas — cada produto vira uma coluna própria.
- PPM: efetivo de espécies diferentes não é somado — cada espécie vira uma coluna própria.
- PIB: valores monetários da PAM e do PIB estão a preços correntes — nenhuma comparação de série temporal é feita sem deixar isso explícito; aqui só integramos os valores, sem deflacionar.
- Períodos diferentes por base (PAM/PPM até 2024, PIB até 2023, VAB agropecuário até 2021) não são forçados a bater — o cruzamento produz linhas com `NaN` fora do período de cada base, e o percentual de correspondência por período é medido explicitamente abaixo.

## Carregamento e recorte ao nível Município

In [2]:
df_pam = pd.read_csv('../data/processed/pam_tratado.csv')
df_ppm = pd.read_csv('../data/processed/ppm_tratado.csv')
df_pib = pd.read_csv('../data/processed/pib_tratado.csv')

pam_mun = df_pam[df_pam['nivel_territorial_nome'] == 'Município'].copy()
ppm_mun = df_ppm[df_ppm['nivel_territorial_nome'] == 'Município'].copy()
pib_mun = df_pib[df_pib['nivel_territorial_nome'] == 'Município'].copy()

for nome, df in [('PAM', pam_mun), ('PPM', ppm_mun), ('PIB', pib_mun)]:
    print(f"{nome}: {df['territorio_codigo'].nunique()} municípios, anos {df['ano_nome'].min()}-{df['ano_nome'].max()}")

PAM: 184 municípios, anos 2003-2024
PPM: 184 municípios, anos 2003-2024
PIB: 184 municípios, anos 2003-2023


## PAM em formato largo (variável × produto)
Chave: `territorio_codigo` + `ano_nome`. Cardinalidade esperada: **1 linha por município-ano** (cada combinação de variável × produto vira uma coluna, nunca somada).

In [3]:
produto_slug = {
    'Milho (em grão)': 'milho',
    'Feijão (em grão)': 'feijao',
    'Mandioca': 'mandioca',
    'Cana-de-açúcar': 'cana',
    'Banana (cacho)': 'banana',
    'Castanha de caju': 'castanha_caju',
    'Melão': 'melao',
}
variavel_slug_pam = {
    'Valor da produção': 'valor',
    'Área plantada ou destinada à colheita': 'area_plantada',
    'Área colhida': 'area_colhida',
    'Quantidade produzida': 'quantidade',
    'Rendimento médio da produção': 'rendimento',
}

pam_mun['coluna'] = (
    'pam_' + pam_mun['variavel_nome'].map(variavel_slug_pam) + '_' + pam_mun['produto_nome'].map(produto_slug)
)

pam_wide = pam_mun.pivot_table(
    index=['territorio_codigo', 'territorio_nome', 'ano_nome'], columns='coluna', values='valor'
).reset_index()

cardinalidade_pam = pam_mun.groupby(['territorio_codigo', 'ano_nome', 'coluna']).size().max()
print(f"Cardinalidade observada (território+ano+coluna -> linhas): máximo {cardinalidade_pam} (esperado 1)")
print(f"Linhas em pam_wide: {len(pam_wide)} (esperado: {pam_mun['territorio_codigo'].nunique()} municípios × {pam_mun['ano_nome'].nunique()} anos = {pam_mun['territorio_codigo'].nunique() * pam_mun['ano_nome'].nunique()})")
pam_wide.head()

Cardinalidade observada (território+ano+coluna -> linhas): máximo 1 (esperado 1)
Linhas em pam_wide: 4048 (esperado: 184 municípios × 22 anos = 4048)


coluna,territorio_codigo,territorio_nome,ano_nome,pam_area_colhida_banana,pam_area_colhida_cana,pam_area_colhida_castanha_caju,pam_area_colhida_feijao,pam_area_colhida_mandioca,pam_area_colhida_melao,pam_area_colhida_milho,pam_area_plantada_banana,pam_area_plantada_cana,pam_area_plantada_castanha_caju,pam_area_plantada_feijao,pam_area_plantada_mandioca,pam_area_plantada_melao,pam_area_plantada_milho,pam_quantidade_banana,pam_quantidade_cana,pam_quantidade_castanha_caju,pam_quantidade_feijao,pam_quantidade_mandioca,pam_quantidade_melao,pam_quantidade_milho,pam_rendimento_banana,pam_rendimento_cana,pam_rendimento_castanha_caju,pam_rendimento_feijao,pam_rendimento_mandioca,pam_rendimento_melao,pam_rendimento_milho,pam_valor_banana,pam_valor_cana,pam_valor_castanha_caju,pam_valor_feijao,pam_valor_mandioca,pam_valor_melao,pam_valor_milho
0,2300101,Abaiara - CE,2003,26.0,100.0,10.0,1300.0,8.0,0.0,2800.0,26.0,100.0,10.0,1300.0,8.0,0.0,2800.0,176.0,5500.0,4.0,702.0,83.0,0.0,5236.0,6769.0,55000.0,400.0,540.0,10375.0,0.0,1870.0,56.0,192.0,4.0,917.0,18.0,0.0,1906.0
1,2300101,Abaiara - CE,2004,26.0,80.0,10.0,1333.0,8.0,0.0,2660.0,26.0,80.0,10.0,1520.0,8.0,0.0,2660.0,169.0,4000.0,5.0,445.0,84.0,0.0,2934.0,6500.0,50000.0,500.0,333.0,10500.0,0.0,1103.0,62.0,152.0,6.0,398.0,16.0,0.0,1141.0
2,2300101,Abaiara - CE,2005,27.0,80.0,10.0,532.0,11.0,0.0,1158.0,27.0,80.0,10.0,532.0,11.0,0.0,1158.0,184.0,3360.0,4.0,117.0,132.0,0.0,903.0,6814.0,42000.0,400.0,219.0,12000.0,0.0,779.0,76.0,119.0,4.0,176.0,12.0,0.0,302.0
3,2300101,Abaiara - CE,2006,35.0,60.0,10.0,750.0,11.0,0.0,1440.0,35.0,60.0,10.0,750.0,11.0,0.0,1440.0,525.0,2400.0,3.0,450.0,132.0,0.0,2880.0,15000.0,40000.0,300.0,600.0,12000.0,0.0,2000.0,297.0,97.0,3.0,401.0,11.0,0.0,877.0
4,2300101,Abaiara - CE,2007,35.0,42.0,10.0,1520.0,11.0,0.0,3810.0,35.0,42.0,10.0,1520.0,11.0,0.0,3810.0,415.0,1400.0,3.0,353.0,117.0,0.0,3225.0,11857.0,33333.0,300.0,232.0,10636.0,0.0,846.0,231.0,58.0,2.0,418.0,21.0,0.0,1329.0


## PPM em formato largo (espécie)
Mesma chave. Cardinalidade esperada: **1 linha por município-ano** (cada espécie vira uma coluna, nunca somada).

In [4]:
especie_slug = {
    'Bovino': 'bovino',
    'Caprino': 'caprino',
    'Ovino': 'ovino',
    'Suíno - total': 'suino',
    'Galináceos - total': 'galinaceos',
}

ppm_mun['coluna'] = 'ppm_efetivo_' + ppm_mun['tipo_rebanho_nome'].map(especie_slug)

ppm_wide = ppm_mun.pivot_table(
    index=['territorio_codigo', 'territorio_nome', 'ano_nome'], columns='coluna', values='valor'
).reset_index()

cardinalidade_ppm = ppm_mun.groupby(['territorio_codigo', 'ano_nome', 'coluna']).size().max()
print(f"Cardinalidade observada: máximo {cardinalidade_ppm} (esperado 1)")
print(f"Linhas em ppm_wide: {len(ppm_wide)}")
ppm_wide.head()

Cardinalidade observada: máximo 1 (esperado 1)
Linhas em ppm_wide: 4048


coluna,territorio_codigo,territorio_nome,ano_nome,ppm_efetivo_bovino,ppm_efetivo_caprino,ppm_efetivo_galinaceos,ppm_efetivo_ovino,ppm_efetivo_suino
0,2300101,Abaiara - CE,2003,5311.0,280.0,25403.0,278.0,539.0
1,2300101,Abaiara - CE,2004,5433.0,290.0,26484.0,285.0,555.0
2,2300101,Abaiara - CE,2005,5615.0,303.0,27762.0,295.0,574.0
3,2300101,Abaiara - CE,2006,5790.0,318.0,28866.0,303.0,587.0
4,2300101,Abaiara - CE,2007,5969.0,333.0,30095.0,310.0,600.0


## PIB em formato largo (variável)
Mesma chave. Cardinalidade esperada: **1 linha por município-ano**.

In [5]:
variavel_slug_pib = {
    'Produto Interno Bruto a preços correntes': 'pib',
    'Valor adicionado bruto a preços correntes total': 'vab_total',
    'Valor adicionado bruto a preços correntes da agropecuária': 'vab_agropecuario',
    'Participação do valor adicionado bruto a preços correntes da agropecuária no valor adicionado bruto a preços correntes total': 'pct_agropecuario_vab',
}

pib_mun['coluna'] = 'pib_' + pib_mun['variavel_nome'].map(variavel_slug_pib)

pib_wide = pib_mun.pivot_table(
    index=['territorio_codigo', 'territorio_nome', 'ano_nome'], columns='coluna', values='valor'
).reset_index()

cardinalidade_pib = pib_mun.groupby(['territorio_codigo', 'ano_nome', 'coluna']).size().max()
print(f"Cardinalidade observada: máximo {cardinalidade_pib} (esperado 1)")
print(f"Linhas em pib_wide: {len(pib_wide)}")
pib_wide.head()

Cardinalidade observada: máximo 1 (esperado 1)
Linhas em pib_wide: 3864


coluna,territorio_codigo,territorio_nome,ano_nome,pib_pct_agropecuario_vab,pib_pib,pib_vab_agropecuario,pib_vab_total
0,2300101,Abaiara - CE,2003,34.46,17657.0,5897.0,17110.0
1,2300101,Abaiara - CE,2004,26.28,16808.0,4235.0,16118.0
2,2300101,Abaiara - CE,2005,18.69,17153.0,3059.0,16367.0
3,2300101,Abaiara - CE,2006,23.04,22869.0,5060.0,21960.0
4,2300101,Abaiara - CE,2007,19.74,25383.0,4831.0,24475.0


## Junção pela chave (território + ano)
`outer join` para preservar todas as combinações e deixar explícito, via `NaN`, onde não há correspondência entre as bases (ex.: PIB não tem 2024; VAB agropecuário não tem 2022-2023).

In [6]:
cruzamento = pam_wide.merge(
    ppm_wide, on=['territorio_codigo', 'territorio_nome', 'ano_nome'], how='outer', suffixes=('', '_ppm')
).merge(
    pib_wide, on=['territorio_codigo', 'territorio_nome', 'ano_nome'], how='outer', suffixes=('', '_pib')
)

cruzamento = cruzamento.sort_values(['territorio_codigo', 'ano_nome']).reset_index(drop=True)
print(f"Linhas no cruzamento final: {len(cruzamento)}")
print(f"Municípios: {cruzamento['territorio_codigo'].nunique()}")
print(f"Anos: {cruzamento['ano_nome'].min()}-{cruzamento['ano_nome'].max()}")
cruzamento.head()

Linhas no cruzamento final: 4048
Municípios: 184
Anos: 2003-2024


coluna,territorio_codigo,territorio_nome,ano_nome,pam_area_colhida_banana,pam_area_colhida_cana,pam_area_colhida_castanha_caju,pam_area_colhida_feijao,pam_area_colhida_mandioca,pam_area_colhida_melao,pam_area_colhida_milho,pam_area_plantada_banana,pam_area_plantada_cana,pam_area_plantada_castanha_caju,pam_area_plantada_feijao,pam_area_plantada_mandioca,pam_area_plantada_melao,pam_area_plantada_milho,pam_quantidade_banana,pam_quantidade_cana,pam_quantidade_castanha_caju,pam_quantidade_feijao,pam_quantidade_mandioca,pam_quantidade_melao,pam_quantidade_milho,pam_rendimento_banana,pam_rendimento_cana,pam_rendimento_castanha_caju,pam_rendimento_feijao,pam_rendimento_mandioca,pam_rendimento_melao,pam_rendimento_milho,pam_valor_banana,pam_valor_cana,pam_valor_castanha_caju,pam_valor_feijao,pam_valor_mandioca,pam_valor_melao,pam_valor_milho,ppm_efetivo_bovino,ppm_efetivo_caprino,ppm_efetivo_galinaceos,ppm_efetivo_ovino,ppm_efetivo_suino,pib_pct_agropecuario_vab,pib_pib,pib_vab_agropecuario,pib_vab_total
0,2300101,Abaiara - CE,2003,26.0,100.0,10.0,1300.0,8.0,0.0,2800.0,26.0,100.0,10.0,1300.0,8.0,0.0,2800.0,176.0,5500.0,4.0,702.0,83.0,0.0,5236.0,6769.0,55000.0,400.0,540.0,10375.0,0.0,1870.0,56.0,192.0,4.0,917.0,18.0,0.0,1906.0,5311.0,280.0,25403.0,278.0,539.0,34.46,17657.0,5897.0,17110.0
1,2300101,Abaiara - CE,2004,26.0,80.0,10.0,1333.0,8.0,0.0,2660.0,26.0,80.0,10.0,1520.0,8.0,0.0,2660.0,169.0,4000.0,5.0,445.0,84.0,0.0,2934.0,6500.0,50000.0,500.0,333.0,10500.0,0.0,1103.0,62.0,152.0,6.0,398.0,16.0,0.0,1141.0,5433.0,290.0,26484.0,285.0,555.0,26.28,16808.0,4235.0,16118.0
2,2300101,Abaiara - CE,2005,27.0,80.0,10.0,532.0,11.0,0.0,1158.0,27.0,80.0,10.0,532.0,11.0,0.0,1158.0,184.0,3360.0,4.0,117.0,132.0,0.0,903.0,6814.0,42000.0,400.0,219.0,12000.0,0.0,779.0,76.0,119.0,4.0,176.0,12.0,0.0,302.0,5615.0,303.0,27762.0,295.0,574.0,18.69,17153.0,3059.0,16367.0
3,2300101,Abaiara - CE,2006,35.0,60.0,10.0,750.0,11.0,0.0,1440.0,35.0,60.0,10.0,750.0,11.0,0.0,1440.0,525.0,2400.0,3.0,450.0,132.0,0.0,2880.0,15000.0,40000.0,300.0,600.0,12000.0,0.0,2000.0,297.0,97.0,3.0,401.0,11.0,0.0,877.0,5790.0,318.0,28866.0,303.0,587.0,23.04,22869.0,5060.0,21960.0
4,2300101,Abaiara - CE,2007,35.0,42.0,10.0,1520.0,11.0,0.0,3810.0,35.0,42.0,10.0,1520.0,11.0,0.0,3810.0,415.0,1400.0,3.0,353.0,117.0,0.0,3225.0,11857.0,33333.0,300.0,232.0,10636.0,0.0,846.0,231.0,58.0,2.0,418.0,21.0,0.0,1329.0,5969.0,333.0,30095.0,310.0,600.0,19.74,25383.0,4831.0,24475.0


## Medindo correspondências e não correspondências
Quanto cada base 'bate' com as outras, por município e por ano.

In [7]:
municipios_pam = set(pam_mun['territorio_codigo'])
municipios_ppm = set(ppm_mun['territorio_codigo'])
municipios_pib = set(pib_mun['territorio_codigo'])

print("--- Correspondência de municípios ---")
print(f"PAM ∩ PPM ∩ PIB: {len(municipios_pam & municipios_ppm & municipios_pib)} de {len(municipios_pam | municipios_ppm | municipios_pib)} municípios totais")
print(f"Em PAM mas não em PPM: {len(municipios_pam - municipios_ppm)}")
print(f"Em PAM mas não em PIB: {len(municipios_pam - municipios_pib)}")
print(f"Em PPM mas não em PIB: {len(municipios_ppm - municipios_pib)}")

--- Correspondência de municípios ---
PAM ∩ PPM ∩ PIB: 184 de 184 municípios totais
Em PAM mas não em PPM: 0
Em PAM mas não em PIB: 0
Em PPM mas não em PIB: 0


In [8]:
anos_pam = set(pam_mun['ano_nome'])
anos_ppm = set(ppm_mun['ano_nome'])
anos_pib = set(pib_mun['ano_nome'])
anos_vab = set(pib_mun.loc[pib_mun['variavel_nome'].str.contains('adicionado'), 'ano_nome'])

print("--- Correspondência de anos ---")
print(f"PAM: {sorted(anos_pam)}")
print(f"PPM: {sorted(anos_ppm)}")
print(f"PIB (todas as variáveis): {sorted(anos_pib)}")
print(f"PIB - VAB (total/agropecuário/participação): {sorted(anos_vab)}")
print()
print(f"PAM ∩ PIB: {len(anos_pam & anos_pib)} de {len(anos_pam)} anos do PAM ({len(anos_pam & anos_pib) / len(anos_pam):.1%})")
print(f"PAM ∩ VAB agropecuário: {len(anos_pam & anos_vab)} de {len(anos_pam)} anos do PAM ({len(anos_pam & anos_vab) / len(anos_pam):.1%})")

--- Correspondência de anos ---
PAM: [2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
PPM: [2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
PIB (todas as variáveis): [2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
PIB - VAB (total/agropecuário/participação): [2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]

PAM ∩ PIB: 21 de 22 anos do PAM (95.5%)
PAM ∩ VAB agropecuário: 21 de 22 anos do PAM (95.5%)


In [9]:
print("--- % de linhas com valor ausente por coluna (no cruzamento final) ---")
colunas_valor = [c for c in cruzamento.columns if c not in ('territorio_codigo', 'territorio_nome', 'ano_nome')]
ausencia = (cruzamento[colunas_valor].isna().mean() * 100).sort_values(ascending=False)
print(ausencia.round(1))

--- % de linhas com valor ausente por coluna (no cruzamento final) ---
coluna
pib_vab_agropecuario               13.6
pib_vab_total                      13.6
pib_pct_agropecuario_vab           13.6
pib_pib                             4.5
pam_area_colhida_mandioca           0.0
pam_area_colhida_melao              0.0
pam_area_colhida_milho              0.0
pam_area_plantada_banana            0.0
pam_area_colhida_banana             0.0
pam_area_colhida_cana               0.0
pam_area_colhida_castanha_caju      0.0
pam_area_colhida_feijao             0.0
pam_area_plantada_mandioca          0.0
pam_area_plantada_feijao            0.0
pam_area_plantada_castanha_caju     0.0
pam_area_plantada_cana              0.0
pam_area_plantada_melao             0.0
pam_area_plantada_milho             0.0
pam_quantidade_banana               0.0
pam_quantidade_cana                 0.0
pam_quantidade_milho                0.0
pam_rendimento_banana               0.0
pam_rendimento_cana                 0.0
pa

## Salvando o cruzamento em `data/analytical/`

In [10]:
import os

os.makedirs('../data/analytical', exist_ok=True)
cruzamento.to_csv('../data/analytical/cruzamento_pam_ppm_pib.csv', index=False)
print(f"Salvo: data/analytical/cruzamento_pam_ppm_pib.csv ({len(cruzamento)} linhas, {len(cruzamento.columns)} colunas)")

Salvo: data/analytical/cruzamento_pam_ppm_pib.csv (4048 linhas, 47 colunas)
